# Cyclistic: how do annual members and casual riders use bikes differently?
Google Data Analytics Capstone, Track A. Each section follows one phase: **Ask → Prepare → Process → Analyze → Share → Act**.

## 1. Ask
- **Business task:** identify how annual members and casual riders use Cyclistic bikes differently, so the marketing team can design a strategy to convert casual riders into annual members.
- **Stakeholders:** Lily Moreno (director of marketing), Cyclistic marketing analytics team, Cyclistic executive team.
- **Key questions:** How do ride length, time of day, day of week, season, bike type and stations differ between the two groups?

## 2. Prepare
12 monthly files of Divvy trip data (see `docs/data_sources.md` for license, ROCCC and limitations).

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW, CLEAN, FIG = ROOT / "data/raw", ROOT / "data/clean", ROOT / "figures"
files = sorted(RAW.glob("*-divvy-tripdata.csv"))
print(len(files), "files:", [f.name for f in files])

In [ ]:
dtypes = {"rideable_type": "category", "member_casual": "category",
          "start_station_id": "string", "end_station_id": "string"}
trips = pd.concat(
    (pd.read_csv(f, dtype=dtypes, parse_dates=["started_at", "ended_at"]) for f in files),
    ignore_index=True,
)
trips.info(memory_usage="deep")

## 3. Process
Every step is recorded in `docs/changelog.md` with row counts before and after.

In [ ]:
log = []
def step(df, name):
    log.append((name, len(df)))
    print(f"{name:<45} {len(df):>10,}")
    return df

df = step(trips, "combined")
df = step(df.drop_duplicates("ride_id"), "drop duplicate ride_id")

df["ride_length_min"] = (df["ended_at"] - df["started_at"]).dt.total_seconds() / 60
df["day_of_week"] = df["started_at"].dt.day_name()
df["month"] = df["started_at"].dt.to_period("M").astype(str)
df["hour"] = df["started_at"].dt.hour

df = step(df[(df["ride_length_min"] > 1) & (df["ride_length_min"] <= 24 * 60)], "ride length in (1 min, 24 h]")
# TODO: inspect and remove test / maintenance stations, check missing values per column
df.isna().mean().sort_values(ascending=False).head(10)

In [ ]:
CLEAN.mkdir(parents=True, exist_ok=True)
df.to_parquet(CLEAN / "trips_clean.parquet", index=False)

## 4. Analyze

In [ ]:
df.groupby("member_casual", observed=True)["ride_length_min"].describe()

In [ ]:
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
by_day = (df.groupby(["member_casual", "day_of_week"], observed=True)
            .agg(rides=("ride_id", "count"), avg_min=("ride_length_min", "mean"))
            .reset_index())
by_day["day_of_week"] = pd.Categorical(by_day["day_of_week"], order, ordered=True)
by_day.sort_values(["member_casual", "day_of_week"])

In [ ]:
by_hour = df.groupby(["member_casual", "hour"], observed=True).size().rename("rides").reset_index()
by_month = df.groupby(["member_casual", "month"], observed=True).size().rename("rides").reset_index()
by_type = pd.crosstab(df["rideable_type"], df["member_casual"], normalize="columns").round(3)
top_casual_stations = (df[df["member_casual"] == "casual"]["start_station_name"]
                       .value_counts().head(10))
by_type, top_casual_stations

## 5. Share
Figures are saved to `figures/` for the slide deck, the README and the portfolio website.

In [ ]:
FIG.mkdir(exist_ok=True)
fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(data=by_day, x="day_of_week", y="rides", hue="member_casual", ax=ax)
ax.set(title="Rides by day of week", xlabel="", ylabel="Rides")
fig.tight_layout(); fig.savefig(FIG / "rides_by_day.png", dpi=200)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
sns.lineplot(data=by_hour, x="hour", y="rides", hue="member_casual", ax=ax)
ax.set(title="Rides by hour of day", xlabel="Hour", ylabel="Rides")
fig.tight_layout(); fig.savefig(FIG / "rides_by_hour.png", dpi=200)

## 6. Act
**Top three recommendations** (each tied to a finding above):
1. …
2. …
3. …

**Further exploration:** pricing data, rider surveys, weather data.